In [ ]:
# https://www.mlit.go.jp/plateau/learning/tpc29-2/

from plateaukit import load_dataset
import pandas as pd
import ipywidgets as widgets
from plateaukit.core.widgets.interactive_deck import InteractiveDeck

dataset = load_dataset("plateau-23100-nagoya-shi-2022")
area = dataset.area_from_landmark("名古屋駅")

river_flooding_risk_df = area.gdf["riverFloodingRisk"].apply(pd.Series)
# river_flooding_risk_df

duration = area.gdf["riverFloodingRisk"].apply(
    lambda x: x["null"]["duration"]
    if isinstance(x, dict)
    and "null" in x
    and isinstance(x["null"], dict)
    and "duration" in x["null"]
    else None
)
area.gdf["riverFloodingRiskDuration"] = duration

# elapsed_hours = 1.0
# in_duration = elapsed_hours <= area.gdf["riverFloodingRiskDuration"]
# in_duration

def show_widget():
    title = widgets.HTML(
        value="<h2>浸水継続時間シミュレーション</h2>"
    )

    slider = widgets.FloatSlider(
        value=0.0,
        min=0,
        max=48.0,
        step=0.01,
        description="0.5m浸水後経過時間（h）",
        layout=widgets.Layout(width="100%"),
    )

    output = widgets.Output()
    deck = InteractiveDeck(area.gdf)
    with output:
        display(deck.deck)

    @output.capture()
    def handler(slider_value):
        gdf = area.gdf.copy()
        elapsed_time = slider_value
        in_duration = elapsed_time <= gdf["riverFloodingRiskDuration"]
        gdf["fill_color"] = in_duration.map({True: [255, 0, 255], False: None})
        updated_layer = deck._create_building_layer(gdf)
        deck.deck.layers = [updated_layer]
        deck.deck.update()

    widgets.interactive(handler, slider_value=slider)

    display(title, slider, output)

show_widget()




HTML(value='<h2>浸水深シミュレーション</h2>')

FloatSlider(value=0.0, description='0.5m浸水後経過時間（h）', layout=Layout(width='100%'), max=48.0, step=0.01)

Output()